# Exercise: Weighted Ensemble Regression for the Debutanizer Column

## Objectives

After this exercise you will be able to:

- build and compare several regression models on the same process dataset
- apply feature scaling before training models that are sensitive to input magnitude
- create a weighted `VotingRegressor` and study how the weights affect performance
- identify useful hyperparameters for model optimization

## Problem Context

The debutanizer column dataset contains seven process variables (`u1` to `u7`) and one target variable (`y`). Your task is to predict `y` using standalone regressors and then improve performance with a weighted ensemble model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error, r2_score

data = pd.read_csv('debutanizer_data.csv')
data.head()

In [ ]:
print('Dataset shape:', data.shape)
display(data.describe())

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(data['y'], bins=30, edgecolor='black')
ax.set_xlabel('y')
ax.set_ylabel('Frequency')
ax.set_title('Target distribution')
plt.show()

## 1. Feature Selection

Use the seven input variables as features and the column `y` as the target. Complete the code cell below.

In [ ]:
# TODO: Build the feature matrix X and target vector y.
# Hint: Use all columns except the target as inputs.
X = ...
y = ...

X.head()

## 2. Train/Test Split and Scaling

Split the data into training and testing sets using `test_size=0.2` and `random_state=42`.

Then scale the features using `StandardScaler`. Scaling is especially important for `SVR` and `MLPRegressor`.

In [ ]:
def evaluate_regression(model_name, y_train_true, y_train_pred, y_test_true, y_test_pred):
    return {
        'Model': model_name,
        'Train MSE': mean_squared_error(y_train_true, y_train_pred),
        'Train R2': r2_score(y_train_true, y_train_pred),
        'Test MSE': mean_squared_error(y_test_true, y_test_pred),
        'Test R2': r2_score(y_test_true, y_test_pred),
    }

# TODO: Split the data, fit the scaler on the training set, and transform both sets.
X_train, X_test, y_train, y_test = ...

scaler = StandardScaler()
X_train_scaled = ...
X_test_scaled = ...

## 3. Standalone Models

Train and compare the following models:

- `LinearRegression`
- `SVR`
- `MLPRegressor`
- `RandomForestRegressor`
- `GradientBoostingRegressor`

Use reasonable starting hyperparameters, then record train/test MSE and R2 in a results table.

In [ ]:
# TODO: Complete the model definitions below.
models = {
    'Linear Regression': LinearRegression(),
    'SVR': SVR(C=..., gamma=..., epsilon=...),
    'ANN': MLPRegressor(
        hidden_layer_sizes=...,
        activation=...,
        max_iter=...,
        alpha=...,
        random_state=42,
    ),
    'Random Forest': RandomForestRegressor(
        n_estimators=...,
        random_state=42,
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=...,
        learning_rate=...,
        random_state=42,
    ),
}

results = []
fitted_models = {}

for name, model in models.items():
    # TODO: Fit each model on the scaled training data, predict on train and test,
    # evaluate it, and store the fitted model.
    ...

results_df = pd.DataFrame(results).sort_values('Test R2', ascending=False)
results_df

## 4. Weighted Ensemble and Hyperparameter Optimization

Create a weighted `VotingRegressor` using a subset of the stronger base models. In a voting regressor, larger weights give a model more influence on the final prediction.

Suggested weight combinations to test:

- `[1, 1, 1, 1]`
- `[1, 2, 2, 2]`
- `[1, 2, 3, 3]`

Leave the optimization work to your own experimentation. You may tune any of the following parameters:

- `SVR`: `kernel`, `C`, `gamma`, `epsilon`
- `MLPRegressor`: `hidden_layer_sizes`, `activation`, `alpha`, `learning_rate_init`, `solver`, `max_iter`
- `RandomForestRegressor`: `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`
- `GradientBoostingRegressor`: `n_estimators`, `learning_rate`, `max_depth`, `subsample`
- `VotingRegressor`: `weights`

Use `GridSearchCV` or `RandomizedSearchCV` if you want a systematic search.

In [ ]:
# TODO: Build a weighted ensemble using fresh estimator definitions.
# Example: use SVR, ANN, Random Forest, and Gradient Boosting in the ensemble.
ensemble_estimators = [
    ('svr', ...),
    ('ann', ...),
    ('rf', ...),
    ('gbr', ...),
]

weights = ...
ensemble = VotingRegressor(estimators=ensemble_estimators, weights=weights)
ensemble.fit(X_train_scaled, y_train)

y_pred_train_ensemble = ...
y_pred_test_ensemble = ...

ensemble_result = evaluate_regression(
    'Weighted Ensemble',
    y_train,
    y_pred_train_ensemble,
    y_test,
    y_pred_test_ensemble,
)

# TODO: Define one or more parameter grids for your optimization study.
svr_param_grid = {
    'kernel': ['rbf', 'poly'],
    'C': [1, 5, 10, 25],
    'gamma': ['scale', 0.1, 1, 5],
    'epsilon': [0.001, 0.01, 0.05, 0.1],
}

mlp_param_grid = {
    'hidden_layer_sizes': [(32, 32), (64, 64), (128, 64)],
    'activation': ['relu', 'tanh'],
    'alpha': [1e-5, 1e-4, 1e-3],
    'learning_rate_init': [0.001, 0.01],
}

# Optional starting point:
# grid_svr = GridSearchCV(SVR(), svr_param_grid, cv=4, scoring='neg_mean_squared_error')
# grid_svr.fit(X_train_scaled, y_train)
# tuned_svr = grid_svr.best_estimator_

# TODO: Compare the weighted ensemble with the standalone models.
# Which model gives the best test performance with a reasonable train-test gap?
...